# Creating a deployment

Let's create a deployment of the `task_maistro` app that we created in module 5.

## Code structure

LangGraph 플랫폼 배포를 생성하려면 다음 정보를 제공해야 합니다:

* [LangGraph API 구성 파일](https://docs.langchain.com/langsmith/application-structure#configuration-file) - `langgraph.json`
* 애플리케이션 로직을 구현하는 그래프 파일 - 예: `task_maistro.py`
* 애플리케이션 실행에 필요한 종속성을 명시하는 파일 - `requirements.txt`
* 애플리케이션 실행에 필요한 환경 변수 제공 - `.env` 또는 `docker-compose.yml`

이 파일들은 이미 `module-6/deployment` 디렉토리에 준비되어 있습니다! 

## CLI

[LangGraph CLI](https://docs.langchain.com/langsmith/cli)는 LangGraph 플랫폼 배포를 생성하기 위한 명령줄 인터페이스입니다.

In [ ]:
# %%capture --no-stderr
# %pip install -U langgraph-cli

To create a <!-- [~self-hosted deployment~](https://langchain-ai.github.io/langgraph/how-tos/deploy-self-hosted/#how-to-do-a-self-hosted-deployment-of-langgraph) --> [self-hosted deployment](https://docs.langchain.com/langsmith/self_hosted_data_plane), we'll follow a few steps. 

### Build Docker Image for LangGraph Server

먼저 langgraph CLI를 사용하여 [LangGraph 서버](https://docs.google.com/presentation/d/18MwIaNR2m4Oba6roK_2VQcBE_8Jq_SI7VHTXJdl7raU/edit#slide=id.g313fb160676_0_32)용 Docker 이미지를 생성합니다.

이를 통해 그래프와 종속성을 Docker 이미지로 패키징합니다.

Docker 이미지는 애플리케이션 실행에 필요한 코드와 종속성을 포함하는 Docker 컨테이너의 템플릿입니다.

[Docker](https://docs.docker.com/engine/install/)가 설치되었는지 확인한 후 다음 명령어를 실행하여 `my-image`라는 Docker 이미지를 생성하세요:

```
$ cd module-6/deployment
$ langgraph build -t my-image
```


### Set Up Redis and PostgreSQL

이미 Redis와 PostgreSQL이 실행 중이라면(예: 로컬 또는 다른 서버에서), LangGraph 서버 컨테이너를 단독으로 생성하고 실행하세요. 이때 Redis와 PostgreSQL의 URI를 지정해야 합니다:

```
docker run \
    --env-file .env \
    -p 8123:8000 \
    -e REDIS_URI="foo" \
    -e DATABASE_URI="bar" \
    -e LANGSMITH_API_KEY="baz" \
    my-image
```

또는 제공된 `docker-compose.yml` 파일을 사용하여 정의된 서비스를 기반으로 세 개의 별도 컨테이너를 생성할 수 있습니다:

* `langgraph-redis`: 공식 Redis 이미지를 사용하여 새 컨테이너를 생성합니다.
* `langgraph-postgres`: 공식 Postgres 이미지를 사용하여 새 컨테이너를 생성합니다.
* `langgraph-api`: 미리 빌드된 이미지를 사용하여 새 컨테이너를 생성합니다.

배포된 `task_maistro` 앱을 실행하려면 `docker-compose-example.yml`을 복사하고 다음 환경 변수를 추가하세요:

* `IMAGE_NAME` (예: `my-image`)
* `LANGSMITH_API_KEY`
* `OPENAI_API_KEY`

그런 다음, <!-- [~배포 시작~](https://langchain-ai.github.io/langgraph/how-tos/deploy-self-hosted/#using-docker-compose) [배포 시작](https://docs.langchain.com/langsmith/self_hosted_data_plane): --> 배포를 시작하세요!

```
$ cd module-6/deployment
$ docker compose up
```